<a href="https://colab.research.google.com/github/Netrahoni/FlyRankAi-Intern-work-Files/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Netrahoni/FlyRankAi-Intern-work-Files/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Row meaning: One row represents one specific anonymized web page at a single point in time.

Tables: We are using the internship-warehouse dataset on Hugging Face, explicitly analyzing a mid-panel month (e.g., 2026-03).

Time window: The trailing 90 days of metrics leading up to the decision point.

Target (Proxy): is_ctr_anomaly (a binary label flagging pages with high impressions but a CTR significantly below the expected average for their ranking position).

Deliberate exclusion: Future clicks or any data from April 2026 onward are excluded to ensure we do not predict the outcome using future knowledge.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
import pandas as pd
from datasets import load_dataset
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

dataset = load_dataset("FlyRank/internship-warehouse", split="train",
                       data_files="*2026-03*.parquet", token=hf_token)
df = dataset.to_pandas()

print("Is grain unique per content_id?:", df['content_id'].is_unique)

print(f"Total rows: {len(df)}")

if 'date' in df.columns:
    print(f"Span: {df['date'].min()} to {df['date'].max()}")

if 'is_active' in df.columns:
    available_df = df[df['is_active'] == True]
    print(f"Rows surviving availability filter: {len(available_df)}")

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Impressions (90d): Knowable at the decision moment because past search volume is permanently logged prior to the rewrite decision.

Average Position: Knowable at the decision moment because historical ranking data is already recorded by search consoles.

Content Age Days: Knowable at the decision moment because the initial publish date is a fixed historical timestamp.

Title Character Length: Knowable at the decision moment because the page's current metadata exists live on the site prior to any updates.

Historical CTR: Knowable at the decision moment because it is calculated directly from trailing impressions and clicks already recorded.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The Trap: Temporarily adding clicks_next_30d (a future metric) as a feature in the training set. This causes the model's accuracy to artificially spike to near-perfect because it is effectively cheating by looking into the future. It is impossible to know future clicks at the moment an editor decides to rewrite a meta tag.

The Limitation: We are relying on historical CTR as a baseline, but changes in Search Engine Results Page (SERP) features (like Google adding a new AI overview) could alter future CTR independently of our meta-tag rewrites.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.